<a href="https://colab.research.google.com/github/luciazhng-web/GeoLife_Trajectory_Mode_Detection/blob/main/4_Baseline%26ModelSelection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This section includes following tasks:
1. Prepare Y and X(select baseline features)
2. Train/Test set split
3. Baseline model training
4. Baseline model evaluation

In [ ]:
import pandas as pd
import ast
import networkx as nx
import matplotlib.pyplot as plt
from os import preadv
#from tqdm import tqdm
import seaborn as sns
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy.stats import mode
from pyproj import CRS, Transformer, Proj
#from google.colab import drive
import geopandas as gpd
import shapely

# Mount Google Drive
#drive.mount('/content/drive')
#DATA_ROOT_DIR = '/content/drive/MyDrive/its_geolife/'
#df_ws = pd.read_csv("/content/drive/MyDrive/its_geolife/df_features.csv",parse_dates=["datetime"],na_values=["NaN"])

In [ ]:
df_ws = pd.read_csv(
    "/home/linuxuser/Downloads/df_features.csv",
    parse_dates=["datetime"],
    na_values=["NaN"],
    dtype={
        "file_name": "string",
        "modes": "category",
        "trip_id": "string",
        "seg_id": "Int64",
        "seg_id_time_split": "string"
    },
    low_memory=False
)

In [ ]:
speed_column = ['datetime','trip_id','modes','dt_sec','speed','is_stop','to_remove','orientation_drop','seg_id','seg_id_time_split','speed_smooth','speed_outlier','speed_filtered','speed_mean','speed_std','speed_skew']
df_ws[(df_ws['dt_sec']>60)&(df_ws['modes'].notna())][speed_column]

,datetime,trip_id,modes,dt_sec,speed,is_stop,to_remove,orientation_drop,seg_id,seg_id_time_split,speed_smooth,speed_outlier,speed_filtered,speed_mean,speed_std,speed_skew
746,2008-12-01 15:50:05,006_00256,bike,27329.0,0.019200,True,False,False,34,34_1,0.0,False,0.0,0.000000,NaN,NaN
14541,2008-12-03 21:58:54,006_00265,bike,10917.0,0.019827,True,True,False,2066,2066_1,0.0,False,0.0,0.000000,NaN,NaN
18228,2008-12-04 17:14:40,006_00266,bike,334.0,0.438394,True,True,True,2526,2526_1,0.0,False,0.0,0.000000,NaN,NaN
36840,2008-12-07 12:18:09,006_00273,bike,1250.0,0.189441,True,False,False,5739,5739_1,0.0,False,0.0,0.000000,NaN,NaN
41064,2009-01-09 08:12:01,006_00312,bike,66.0,0.079604,True,False,False,6391,6391_1,0.0,False,0.0,0.000000,0.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5737082,2008-04-18 21:17:41,031_00128,bike,240.0,4.832575,False,False,False,614667,614667_1,0.0,False,0.0,3.286057,0.988053,-0.633992
5737187,2008-04-19 01:40:07,031_00129,bike,15521.0,0.028352,True,True,True,614667,614667_2,0.0,False,0.0,0.000000,NaN,NaN
5737612,2008-04-19 10:30:02,031_00130,walk,30944.0,0.143437,True,False,False,614669,614669_1,0.0,False,0.0,0.017132,0.024228,0.000000
5737614,2008-04-19 10:34:03,031_00130,walk,240.0,0.200830,True,False,False,614669,614669_2,0.0,False,0.0,0.000000,NaN,NaN


In [ ]:
df_ws[df_ws['seg_id']==34][speed_column]

,datetime,trip_id,modes,dt_sec,speed,is_stop,to_remove,orientation_drop,seg_id,seg_id_time_split,speed_smooth,speed_outlier,speed_filtered,speed_mean,speed_std,speed_skew
695,2008-12-01 08:12:48,<NA>,NaN,2.0,0.759617,True,False,False,34,34_0,0.0,False,0.0,0.0,0.0,NaN
696,2008-12-01 08:12:50,<NA>,NaN,2.0,0.591574,True,False,False,34,34_0,0.0,False,0.0,0.0,0.0,NaN
697,2008-12-01 08:12:52,<NA>,NaN,2.0,0.520425,True,False,False,34,34_0,0.0,False,0.0,0.0,0.0,NaN
698,2008-12-01 08:12:54,<NA>,NaN,2.0,0.367370,True,False,False,34,34_0,0.0,False,0.0,0.0,0.0,NaN
699,2008-12-01 08:12:56,<NA>,NaN,2.0,0.367370,True,False,False,34,34_0,0.0,False,0.0,0.0,0.0,NaN
700,2008-12-01 08:12:58,<NA>,NaN,2.0,0.367370,True,False,False,34,34_0,0.0,False,0.0,0.0,0.0,NaN
701,2008-12-01 08:13:00,<NA>,NaN,2.0,0.290503,True,False,False,34,34_0,0.0,False,0.0,0.0,0.0,NaN
702,2008-12-01 08:13:02,<NA>,NaN,2.0,0.295787,True,False,False,34,34_0,0.0,False,0.0,0.0,0.0,NaN
703,2008-12-01 08:13:04,<NA>,NaN,2.0,0.511418,True,False,False,34,34_0,0.0,False,0.0,0.0,0.0,NaN
704,2008-12-01 08:13:06,<NA>,NaN,2.0,0.511418,True,False,False,34,34_0,0.0,False,0.0,0.0,0.0,NaN


In [ ]:
df_ws = df_ws[df_ws['modes'].notna()]
len(df_ws)

3311269

In [ ]:
df_ws.columns

Index(['idx_o', 'datetime', 'file_name', 'modes', 'trip_id', 'dt_sec',
       'heading', 'distance', 'alt_m', 'speed', 'to_remove', 'is_stop',
       'seg_id', 'orientation_drop', 'seg_id_time_split', 'x_smooth',
       'y_smooth', 'vx_smooth', 'vy_smooth', 'alt_m_smooth', 'vz_smooth',
       'speed_smooth', 'distance_smooth', 'heading_smooth', 'accel',
       'ver_speed', 'alt_outlier', 'speed_filtered', 'accel_filtered',
       'vspeed_filtered', 'speed_outlier', 'speed_mean', 'speed_std',
       'speed_skew', 'speed_kurt', 'speed_p10', 'speed_p30', 'speed_p50',
       'speed_p70', 'speed_p95', 'accel_mean', 'accel_std', 'accel_skew',
       'accel_kurt', 'accel_p10', 'accel_p30', 'accel_p50', 'accel_p70',
       'accel_p95', 'yaw', 'yaw_mean', 'yaw_std', 'yaw_skew', 'yaw_kurt',
       'yaw_p10', 'yaw_p30', 'yaw_p50', 'yaw_p70', 'yaw_p95', 'SR',
       'straight_rate', 'dist_bus', 'dist_metro', 'bus_mean', 'bus_std',
       'bus_skew', 'bus_kurt', 'bus_p10', 'bus_p30', 'bus_p50', 'bu

### Feature selections
1. Raw data
2. Basic features
3. Enriched features
4. optional GIS features

In [ ]:
# Raw data
raw_features = ['dt_sec','heading','distance','alt_m','speed']

# Basic features
basic_features = ['dt_sec', 'to_remove', 'is_stop', 'orientation_drop', 'speed_filtered',
        'accel_filtered','vspeed_filtered','heading_smooth','alt_m_smooth','speed_outlier','alt_outlier']

# Enriched features
enriched_features = ['dt_sec', 'to_remove', 'is_stop', 'orientation_drop', 'speed_filtered',
       'accel_filtered','vspeed_filtered','heading_smooth','alt_m_smooth','speed_outlier',
       'alt_outlier','speed_mean', 'speed_std', 'speed_skew', 'speed_kurt', 'speed_p10',
       'speed_p30', 'speed_p50', 'speed_p70', 'speed_p95', 'accel_mean',
       'accel_std', 'accel_skew', 'accel_kurt', 'accel_p10', 'accel_p30',
       'accel_p50', 'accel_p70', 'accel_p95', 'yaw', 'yaw_mean', 'yaw_std',
       'yaw_skew', 'yaw_kurt', 'yaw_p10', 'yaw_p30', 'yaw_p50', 'yaw_p70',
       'yaw_p95', 'SR', 'straight_rate']

# GIS features
gis_features = ['dt_sec', 'to_remove', 'is_stop', 'orientation_drop', 'speed_filtered',
       'accel_filtered','vspeed_filtered','heading_smooth','alt_m_smooth','speed_outlier',
       'alt_outlier','speed_mean', 'speed_std', 'speed_skew', 'speed_kurt', 'speed_p10',
       'speed_p30', 'speed_p50', 'speed_p70', 'speed_p95', 'accel_mean',
       'accel_std', 'accel_skew', 'accel_kurt', 'accel_p10', 'accel_p30',
       'accel_p50', 'accel_p70', 'accel_p95', 'yaw', 'yaw_mean', 'yaw_std',
       'yaw_skew', 'yaw_kurt', 'yaw_p10', 'yaw_p30', 'yaw_p50', 'yaw_p70',
       'yaw_p95', 'SR', 'straight_rate', 'dist_bus', 'dist_metro', 'bus_mean', 'bus_std',
       'bus_skew', 'bus_kurt', 'bus_p10', 'bus_p30', 'bus_p50', 'bus_p70',
       'bus_p95', 'metro_mean', 'metro_std', 'metro_skew', 'metro_kurt',
       'metro_p10', 'metro_p30', 'metro_p50', 'metro_p70', 'metro_p95',
       'ratio_bus', 'ratio_metro']

## Train/Test split by Segment

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Prepare groups by trip_id to avoid data leakage (using seg_id or seg_id_time_split will have data leakage)
groups = df_ws['trip_id'].astype(str)

# Define X and y
X = df_ws  # Example features
y = df_ws['modes']  # Target variable

# Group-based train/val split
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
for train_idx, val_idx in gss.split(X, y, groups=groups):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

In [ ]:
y_train.value_counts()

walk        844846
bus         577834
car         451389
bike        368094
subway      136959
train       102171
airplane      4988
Name: modes, dtype: int64

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc  = le.transform(y_val)

dict(zip(le.classes_, le.transform(le.classes_)))

{'airplane': 0,
 'bike': 1,
 'bus': 2,
 'car': 3,
 'subway': 4,
 'train': 5,
 'walk': 6}

## Baseline model - GXBoost

In [ ]:
from pandas import read_csv
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
import graphviz

from sklearn.tree import export_graphviz
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.ensemble import RandomForestClassifier,VotingClassifier,GradientBoostingClassifier,StackingClassifier,ExtraTreesClassifier
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.naive_bayes import GaussianNB

import warnings
warnings.filterwarnings('ignore')

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc  = le.transform(y_val)

dict(zip(le.classes_, le.transform(le.classes_)))

{'airplane': 0,
 'bike': 1,
 'bus': 2,
 'car': 3,
 'subway': 4,
 'train': 5,
 'walk': 6}

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = le.classes_

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

# Map weights to encoded labels
weight_map = dict(zip(le.transform(classes), weights))

sample_weight = [weight_map[k] for k in y_train_enc]

In [ ]:
le.transform(classes)

array([0, 1, 2, 3, 4, 5, 6])

In [ ]:
len(sample_weight)

2486281

In [ ]:
def compute_metrics(y_true,y_pred):
    accuracy = accuracy_score(y_true,y_pred)
    f1_score_macro = f1_score(y_true,y_pred,average='macro')
    f1_score_weighted = f1_score(y_true,y_pred,average='weighted')
    return [accuracy,f1_score_macro,f1_score_weighted]

results = pd.DataFrame(columns=['Accuracy','F1-score (macro avg)', 'F1-score(weighted avg)'])

Baseline model - raw features

In [ ]:
!pip install xgboost
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',
    num_class=len(le.classes_),
    tree_method='hist',
    eval_metric='mlogloss'
)

model.fit(
    X_train[raw_features],
    y_train_enc,
    sample_weight=sample_weight
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_class=7, num_parallel_tree=None, ...)

In [ ]:
from sklearn.metrics import classification_report, f1_score, confusion_matrix

y_pred_enc = model.predict(X_val[raw_features])
y_pred = le.inverse_transform(y_pred_enc)
results.loc['Raw features',:] = compute_metrics(y_val,y_pred)
print(classification_report(y_val, y_pred))
print("Macro F1:", f1_score(y_val, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_val, y_pred, average="weighted"))

              precision    recall  f1-score   support

    airplane       0.18      0.73      0.28       991
        bike       0.52      0.70      0.60    122085
         bus       0.58      0.36      0.45    205797
         car       0.67      0.51      0.58    146069
      subway       0.39      0.62      0.48     50227
       train       0.38      0.53      0.44     31545
        walk       0.65      0.71      0.68    268274

    accuracy                           0.57    824988
   macro avg       0.48      0.60      0.50    824988
weighted avg       0.59      0.57      0.57    824988

Macro F1: 0.5011001032244211
Weighted F1: 0.5693967638761219


In [ ]:
results.sort_values(by='F1-score(weighted avg)', ascending=False)

,Accuracy,F1-score (macro avg),F1-score(weighted avg)
Raw features,0.574485,0.5011,0.569397


Balance 'airplane' mode data

In [ ]:
df_ws['modes'].value_counts()

,count
modes,
walk,1113120
bus,783631
car,597458
bike,490179
subway,187186
train,133716
airplane,5979


In [ ]:
list_airplane = [59953,417095,419675,492876,500863]

In [ ]:
def mode_balance(df, id_list):
  df = df.copy()
  for idx, seg_id in enumerate(id_list):
    mask = (df['seg_id']==seg_id)
    df.loc[mask, 'modes'] = 'airplane'
    unique_trip_id = f"'enriched_airplane_'+{idx:03d}"
    df.loc[mask, 'trip_id'] = unique_trip_id
  return df

In [ ]:
df_balanced  = mode_balance(df_ws, list_airplane)

In [ ]:
df_balanced = df_balanced[df_balanced['modes'].notna()]

In [ ]:
df_balanced['modes'].value_counts()

,count
modes,
walk,1113120
bus,783631
car,597458
bike,490179
subway,187186
train,133716
airplane,58112


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Prepare groups by trip_id to avoid data leakage (using seg_id or seg_id_time_split will have data leakage)
groups = df_balanced['trip_id'].astype(str)

# Define X and y
X = df_balanced  # Example features
y = df_balanced['modes']  # Target variable

# Group-based train/val split
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
for train_idx, val_idx in gss.split(X, y, groups=groups):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

In [ ]:
y_train.value_counts()

,count
modes,
walk,875108
bus,568606
car,431315
bike,361706
subway,144205
train,89673
airplane,55654


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc  = le.transform(y_val)

dict(zip(le.classes_, le.transform(le.classes_)))

{'airplane': np.int64(0),
 'bike': np.int64(1),
 'bus': np.int64(2),
 'car': np.int64(3),
 'subway': np.int64(4),
 'train': np.int64(5),
 'walk': np.int64(6)}

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = le.classes_

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

# Map weights to encoded labels
weight_map = dict(zip(le.transform(classes), weights))

sample_weight = [weight_map[k] for k in y_train_enc]

In [ ]:
sample_weight

Re-run the baseline model with raw features in balanced dataset

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',
    num_class=len(le.classes_),
    tree_method='hist',
    eval_metric='mlogloss'
)

model.fit(
    X_train[raw_features],
    y_train_enc,
    sample_weight=sample_weight
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=7, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None, num_class=7, ...)

In [ ]:
from sklearn.metrics import classification_report, f1_score, confusion_matrix

y_pred_enc = model.predict(X_val[raw_features])
y_pred = le.inverse_transform(y_pred_enc)
results.loc['Raw features_balanced',:] = compute_metrics(y_val,y_pred)
print(classification_report(y_val, y_pred))
print("Macro F1:", f1_score(y_val, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_val, y_pred, average="weighted"))

              precision    recall  f1-score   support

    airplane       0.82      0.72      0.77      2458
        bike       0.52      0.68      0.59    128473
         bus       0.61      0.34      0.44    215025
         car       0.70      0.50      0.58    166143
      subway       0.31      0.66      0.42     42981
       train       0.49      0.49      0.49     44043
        walk       0.58      0.70      0.63    238012

    accuracy                           0.55    837135
   macro avg       0.57      0.58      0.56    837135
weighted avg       0.58      0.55      0.55    837135

Macro F1: 0.5597863106013452
Weighted F1: 0.5479840033774872


In [ ]:
results.sort_values(by='F1-score(weighted avg)', ascending=False)

,Accuracy,F1-score (macro avg),F1-score(weighted avg)
Raw features_balanced,0.55308,0.559786,0.547984
Raw features,0.574757,0.500844,0.569756


The results indicates oversample, so the final strategy is to drop airplane mode

In [ ]:
df_drop = df_ws[df_ws['modes']!='airplane']

In [ ]:
df_drop = df_drop[df_drop['modes'].notna()]

In [ ]:
df_drop['modes'].value_counts()

walk        1113120
bus          783631
car          597458
bike         490179
subway       187186
train        133716
airplane          0
Name: modes, dtype: int64

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Prepare groups by trip_id to avoid data leakage (using seg_id or seg_id_time_split will have data leakage)
groups = df_drop['trip_id'].astype(str)

# Define X and y
X = df_drop  # Example features
y = df_drop['modes']  # Target variable

# Group-based train/val split
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
for train_idx, val_idx in gss.split(X, y, groups=groups):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

In [ ]:
y_train.value_counts()

walk        814233
bus         599491
car         456818
bike        379457
subway      144372
train       110366
airplane         0
Name: modes, dtype: int64

In [ ]:
y_val.value_counts()

walk        298887
bus         184140
car         140640
bike        110722
subway       42814
train        23350
airplane         0
Name: modes, dtype: int64

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc  = le.transform(y_val)

dict(zip(le.classes_, le.transform(le.classes_)))

{'bike': 0, 'bus': 1, 'car': 2, 'subway': 3, 'train': 4, 'walk': 5}

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = le.classes_

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

# Map weights to encoded labels
weight_map = dict(zip(le.transform(classes), weights))

sample_weight = [weight_map[k] for k in y_train_enc]

NameError: name 'y_train' is not defined

In [ ]:
len(sample_weight)

2504737

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',
    num_class=len(le.classes_),
    tree_method='hist',
    eval_metric='mlogloss'
)

model.fit(
    X_train[raw_features],
    y_train_enc,
    sample_weight=sample_weight
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_class=6, num_parallel_tree=None, ...)

In [ ]:
from sklearn.metrics import classification_report, f1_score, confusion_matrix

y_pred_enc = model.predict(X_val[raw_features])
y_pred = le.inverse_transform(y_pred_enc)
results.loc['Raw features_drop',:] = compute_metrics(y_val,y_pred)
print(classification_report(y_val, y_pred))
print("Macro F1:", f1_score(y_val, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_val, y_pred, average="weighted"))

              precision    recall  f1-score   support

        bike       0.49      0.72      0.58    110722
         bus       0.57      0.36      0.44    184140
         car       0.69      0.48      0.57    140640
      subway       0.38      0.58      0.46     42814
       train       0.30      0.46      0.36     23350
        walk       0.68      0.73      0.71    298887

    accuracy                           0.59    800553
   macro avg       0.52      0.56      0.52    800553
weighted avg       0.61      0.59      0.58    800553

Macro F1: 0.5196538244545827
Weighted F1: 0.5809067581939751


In [ ]:
results.sort_values(by='F1-score(weighted avg)', ascending=False)

,Accuracy,F1-score (macro avg),F1-score(weighted avg)
Raw features_drop,0.585449,0.519654,0.580907
Raw features,0.574485,0.5011,0.569397


Baseline model - Basic features

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',
    num_class=len(le.classes_),
    tree_method='hist',
    eval_metric='mlogloss'
)

model.fit(
    X_train[basic_features],
    y_train_enc,
    sample_weight=sample_weight
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_class=6, num_parallel_tree=None, ...)

In [ ]:
from sklearn.metrics import classification_report, f1_score, confusion_matrix

y_pred_enc = model.predict(X_val[basic_features])
y_pred = le.inverse_transform(y_pred_enc)
results.loc['Basic features',:] = compute_metrics(y_val,y_pred)
print(classification_report(y_val, y_pred))
print("Macro F1:", f1_score(y_val, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_val, y_pred, average="weighted"))

              precision    recall  f1-score   support

        bike       0.59      0.76      0.66    110722
         bus       0.62      0.44      0.51    184140
         car       0.70      0.49      0.58    140640
      subway       0.42      0.64      0.50     42814
       train       0.38      0.53      0.44     23350
        walk       0.70      0.78      0.74    298887

    accuracy                           0.63    800553
   macro avg       0.57      0.61      0.57    800553
weighted avg       0.64      0.63      0.63    800553

Macro F1: 0.5731116962491706
Weighted F1: 0.6260771717577952


In [ ]:
results.sort_values(by='F1-score(weighted avg)', ascending=False)

,Accuracy,F1-score (macro avg),F1-score(weighted avg)
Basic features,0.631464,0.573112,0.626077
Raw features_drop,0.585449,0.519654,0.580907
Raw features,0.574485,0.5011,0.569397


Baseline Model - Enriched features

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',
    num_class=len(le.classes_),
    tree_method='hist',
    eval_metric='mlogloss'
)

model.fit(
    X_train[enriched_features],
    y_train_enc,
    sample_weight=sample_weight
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_class=6, num_parallel_tree=None, ...)

In [ ]:
from sklearn.metrics import classification_report, f1_score, confusion_matrix

y_pred_enc = model.predict(X_val[enriched_features])
y_pred = le.inverse_transform(y_pred_enc)
results.loc['Enriched features',:] = compute_metrics(y_val,y_pred)
print(classification_report(y_val, y_pred))
print("Macro F1:", f1_score(y_val, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_val, y_pred, average="weighted"))

              precision    recall  f1-score   support

        bike       0.75      0.82      0.79    110722
         bus       0.75      0.66      0.70    184140
         car       0.77      0.68      0.72    140640
      subway       0.68      0.76      0.72     42814
       train       0.81      0.75      0.78     23350
        walk       0.78      0.85      0.81    298887

    accuracy                           0.76    800553
   macro avg       0.76      0.75      0.75    800553
weighted avg       0.76      0.76      0.76    800553

Macro F1: 0.7528425595346707
Weighted F1: 0.7605766902235286


In [ ]:
results.sort_values(by='F1-score (macro avg)', ascending=False)

,Accuracy,F1-score (macro avg),F1-score(weighted avg)
Enriched features,0.762645,0.752843,0.760577
Basic features,0.631464,0.573112,0.626077
Raw features_drop,0.585449,0.519654,0.580907
Raw features,0.574485,0.5011,0.569397


Baseline model - GIS features

In [ ]:
model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',
    num_class=len(le.classes_),
    tree_method='hist',
    eval_metric='mlogloss'
)

model.fit(
    X_train[gis_features],
    y_train_enc,
    sample_weight=sample_weight
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_class=6, num_parallel_tree=None, ...)

In [ ]:
y_pred_enc = model.predict(X_val[gis_features])
y_pred = le.inverse_transform(y_pred_enc)
results.loc['GIS features',:] = compute_metrics(y_val,y_pred)
print(classification_report(y_val, y_pred))
print("Macro F1:", f1_score(y_val, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_val, y_pred, average="weighted"))

              precision    recall  f1-score   support

        bike       0.79      0.82      0.80    110722
         bus       0.73      0.72      0.73    184140
         car       0.78      0.71      0.74    140640
      subway       0.72      0.82      0.76     42814
       train       0.79      0.77      0.78     23350
        walk       0.80      0.81      0.80    298887

    accuracy                           0.77    800553
   macro avg       0.77      0.77      0.77    800553
weighted avg       0.77      0.77      0.77    800553

Macro F1: 0.7700532581325051
Weighted F1: 0.7726756156623954


In [ ]:
results.sort_values(by='F1-score (macro avg)', ascending=False)

,Accuracy,F1-score (macro avg),F1-score(weighted avg)
GIS features,0.773129,0.770053,0.772676
Enriched features,0.762645,0.752843,0.760577
Basic features,0.631464,0.573112,0.626077
Raw features_drop,0.585449,0.519654,0.580907
Raw features,0.574485,0.5011,0.569397


In [ ]:
importance_score = model.get_booster().get_score(importance_type='gain')

{'dt_sec': 303.2734069824219,
 'to_remove': 80.94840240478516,
 'is_stop': 74.02559661865234,
 'orientation_drop': 16.533037185668945,
 'speed_filtered': 38.5004997253418,
 'accel_filtered': 11.939878463745117,
 'vspeed_filtered': 85.19066619873047,
 'heading_smooth': 104.52941131591797,
 'alt_m_smooth': 229.28709411621094,
 'speed_outlier': 1.197906494140625,
 'alt_outlier': 29.048662185668945,
 'speed_mean': 721.8943481445312,
 'speed_std': 318.38006591796875,
 'speed_skew': 121.34841918945312,
 'speed_kurt': 113.39351654052734,
 'speed_p10': 1062.2100830078125,
 'speed_p30': 1375.622314453125,
 'speed_p50': 1303.1744384765625,
 'speed_p70': 1530.1439208984375,
 'speed_p95': 1068.19091796875,
 'accel_mean': 385.6733093261719,
 'accel_std': 173.905029296875,
 'accel_skew': 326.0506286621094,
 'accel_kurt': 130.1693572998047,
 'accel_p10': 181.75535583496094,
 'accel_p30': 198.19493103027344,
 'accel_p50': 157.4977264404297,
 'accel_p70': 733.8743286132812,
 'accel_p95': 554.3936157226

In [ ]:
importance_df = pd.DataFrame.from_dict(importance_score, orient='index', columns=['score'])
importance_df

,score
dt_sec,6.083948
to_remove,1.315914
is_stop,0.696639
orientation_drop,0.582865
speed_filtered,0.329355
...,...
metro_p50,4.090549
metro_p70,34.844143
metro_p95,46.861378
ratio_bus,2.767429


In [ ]:
importance_df.sort_values(by='score', ascending=False).head(15)

,score
metro_p95,46.861378
metro_p70,34.844143
speed_p30,33.236717
speed_p50,27.349321
speed_p70,26.208721
speed_p95,20.561934
metro_mean,19.867838
speed_p10,18.882917
speed_mean,18.531488
metro_p10,15.185461


## Model Selection

Hyper parameter tuning

In [ ]:
from xgboost import XGBClassifier

from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.metrics import make_scorer, f1_score

params_1 = {
    'max_depth': [10,15,20,25,30],
    'learning_rate': [0.01, 0.05, 0.1,0.15]
}

cv = GroupKFold(n_splits=3)

groups = X_train['trip_id'].astype(str)

XGB_3CV = GridSearchCV(
    XGBClassifier(
    n_estimators=300,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1,
    objective='multi:softmax',
    num_class=len(le.classes_),
    tree_method='hist',
    eval_metric='mlogloss'),
    param_grid=params_1,
    cv=cv,
    verbose=2,
    n_jobs=1
)
XGB_3CV.fit(X_train[gis_features],
    y_train_enc,
    groups=groups,
    sample_weight=sample_weight)
print("Best params:", XGB_3CV.best_params_)

/home/linuxuser/.local/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


Fitting 3 folds for each of 20 candidates, totalling 60 fits


Exception ignored on calling ctypes callback function: <bound method DataIter._next_wrapper of <xgboost.data.SingleBatchInternalIter object at 0x7fdcd9f70d60>>
Traceback (most recent call last):
  File "/home/linuxuser/.local/lib/python3.8/site-packages/xgboost/core.py", line 594, in _next_wrapper
    def _next_wrapper(self, this: None) -> int:  # pylint: disable=unused-argument
KeyboardInterrupt: 


[CV] END ...................learning_rate=0.01, max_depth=10; total time=   1.5s


KeyboardInterrupt: 

In [ ]:
results = XGB_3CV.cv_results_
results

{'mean_fit_time': array([199.91418783, 359.38573805, 601.44135722, 817.19212802,
        960.41292349, 174.14370664, 319.11383597, 463.9094007 ,
        550.14704498, 594.98616656, 170.91315126, 299.70649695,
        385.69367393, 430.87357203, 455.18297219, 170.5212551 ,
        282.1123666 , 345.94521626, 380.2480127 , 398.25528781]),
 'std_fit_time': array([ 7.18513007,  5.80497874, 17.4218379 , 24.67149702, 22.9618464 ,
         1.04855602,  2.62819124, 11.00578899, 19.09464239, 21.48985343,
         1.61299204,  7.79666972, 16.33245263, 22.75536197, 25.16596052,
         0.29076311, 10.14990472, 19.7131753 , 25.15527517, 28.20551523]),
 'mean_score_time': array([ 7.73768536, 14.17991567, 19.21588349, 22.66894269, 24.75724204,
         7.31323147, 13.57563599, 18.09177836, 20.38350542, 21.39125776,
         7.26270429, 13.67601228, 17.19907395, 18.77958703, 19.65436546,
         7.28224619, 13.341338  , 16.48483841, 17.90523394, 18.4880499 ]),
 'std_score_time': array([0.35375245, 

In [ ]:
from xgboost import XGBClassifier

from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.metrics import make_scorer, f1_score


params_2 = {
    'subsample': [0.7,0.8, 0.9],
    'colsample_bytree': [0.6,0.8,0.9],
    'reg_lambda': [1, 5, 10]
}

cv = GroupKFold(n_splits=3)

groups = X_train['trip_id'].astype(str)

XGB_3CV = GridSearchCV(
    XGBClassifier(
    n_estimators=300,
    max_depth=25,
    learning_rate=0.1,
    objective='multi:softmax',
    num_class=len(le.classes_),
    tree_method='hist',
    eval_metric='mlogloss'),
    param_grid=params_2,
    cv=cv,
    verbose=2,
    n_jobs=1
)
XGB_3CV.fit(X_train[gis_features],
    y_train_enc,
    groups=groups,
    sample_weight=sample_weight)
print("Best params:", XGB_3CV.best_params_)

In [ ]:
from xgboost import XGBClassifier

from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.metrics import make_scorer, f1_score

params_3 = {
    'n_estimators': [300,350,400,450,500]
    }

cv = GroupKFold(n_splits=3)

groups = X_train['trip_id'].astype(str)

XGB_3CV = GridSearchCV(
    XGBClassifier(
    max_depth=25,
    learning_rate=0.1,
    subsample=0.7,
    colsample_bytree=0.6,
    reg_lambda=10,
    objective='multi:softmax',
    num_class=len(le.classes_),
    tree_method='hist',
    eval_metric='mlogloss'),
    param_grid=params_3,
    cv=cv,
    verbose=2,
    n_jobs=1
)
XGB_3CV.fit(X_train[gis_features],
    y_train_enc,
    groups=groups,
    sample_weight=sample_weight)
print("Best params:", XGB_3CV.best_params_)

Fitting 3 folds for each of 5 candidates, totalling 15 fits
[CV] END ...................................n_estimators=300; total time= 8.5min
[CV] END ...................................n_estimators=300; total time= 8.0min
[CV] END ...................................n_estimators=300; total time= 8.8min
[CV] END ...................................n_estimators=350; total time= 9.3min
[CV] END ...................................n_estimators=350; total time= 9.0min
[CV] END ...................................n_estimators=350; total time= 9.9min
[CV] END ...................................n_estimators=400; total time=10.3min
[CV] END ...................................n_estimators=400; total time= 9.8min
[CV] END ...................................n_estimators=400; total time=11.0min
[CV] END ...................................n_estimators=450; total time=11.2min
[CV] END ...................................n_estimators=450; total time=10.7min
[CV] END ...................................n_est

In [ ]:
from xgboost import XGBClassifier

model  = XGBClassifier(
    n_estimators=350,
    learning_rate=0.1,
    max_depth=25,
    subsample=0.7,
    colsample_bytree=0.6,
    reg_lambda=10,
    objective='multi:softmax',
    num_class=len(le.classes_),
    tree_method='hist',
    eval_metric='mlogloss'
)

model.fit(
    X_train[gis_features],
    y_train_enc,
    sample_weight=sample_weight
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.6, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=25,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=350,
              n_jobs=None, num_class=6, num_parallel_tree=None, ...)

In [ ]:
from sklearn.metrics import classification_report, f1_score, confusion_matrix
y_pred_enc = model.predict(X_val[gis_features])
y_pred = le.inverse_transform(y_pred_enc)
results.loc['Final Model',:] = compute_metrics(y_val,y_pred)
print(classification_report(y_val, y_pred))
print("Macro F1:", f1_score(y_val, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_val, y_pred, average="weighted"))

              precision    recall  f1-score   support

        bike       0.86      0.81      0.83    110722
         bus       0.76      0.73      0.74    184140
         car       0.81      0.70      0.75    140640
      subway       0.91      0.71      0.80     42814
       train       0.80      0.83      0.82     23350
        walk       0.77      0.88      0.82    298887

    accuracy                           0.79    800553
   macro avg       0.82      0.78      0.79    800553
weighted avg       0.80      0.79      0.79    800553

Macro F1: 0.7946954784649997
Weighted F1: 0.7919846559608339


In [ ]:
results.sort_values(by='F1-score (macro avg)', ascending=False)

,Accuracy,F1-score (macro avg),F1-score(weighted avg)
Final Model,0.793272,0.794695,0.791985
GIS features,0.773129,0.770053,0.772676
Enriched features,0.762645,0.752843,0.760577
Basic features,0.631464,0.573112,0.626077
Raw features_drop,0.585449,0.519654,0.580907
Raw features,0.574485,0.5011,0.569397


In [ ]:
importance_score = model.get_booster().get_score(importance_type='gain')

In [ ]:
importance_df = pd.DataFrame.from_dict(importance_score, orient='index', columns=['score'])
importance_df.sort_values(by='score', ascending=False).head(15)

,score
metro_p95,46.861378
metro_p70,34.844143
speed_p30,33.236717
speed_p50,27.349321
speed_p70,26.208721
speed_p95,20.561934
metro_mean,19.867838
speed_p10,18.882917
speed_mean,18.531488
metro_p10,15.185461


In [ ]:
importance_score

{'dt_sec': 6.083948135375977,
 'to_remove': 1.3159136772155762,
 'is_stop': 0.6966390013694763,
 'orientation_drop': 0.5828649997711182,
 'speed_filtered': 0.3293549418449402,
 'accel_filtered': 0.21101324260234833,
 'vspeed_filtered': 0.40743404626846313,
 'heading_smooth': 0.5221074819564819,
 'alt_m_smooth': 2.700115919113159,
 'speed_outlier': 0.15904639661312103,
 'alt_outlier': 4.481618881225586,
 'speed_mean': 18.5314884185791,
 'speed_std': 9.835136413574219,
 'speed_skew': 2.829622983932495,
 'speed_kurt': 2.3126933574676514,
 'speed_p10': 18.882917404174805,
 'speed_p30': 33.236717224121094,
 'speed_p50': 27.349321365356445,
 'speed_p70': 26.208721160888672,
 'speed_p95': 20.561933517456055,
 'accel_mean': 4.288961410522461,
 'accel_std': 2.8705403804779053,
 'accel_skew': 4.40344762802124,
 'accel_kurt': 3.25062894821167,
 'accel_p10': 2.477323532104492,
 'accel_p30': 2.4749226570129395,
 'accel_p50': 2.6542389392852783,
 'accel_p70': 8.775607109069824,
 'accel_p95': 7.04922

In [ ]:
len(df_ws['trip_id'].unique())

6112

In [ ]:
len(X_train['trip_id'].unique())

4578

In [ ]:
len(X_val['trip_id'].unique())

1527

In [ ]:
y_train.value_counts()

walk        814233
bus         599491
car         456818
bike        379457
subway      144372
train       110366
airplane         0
Name: modes, dtype: int64

In [ ]:
y_val.value_counts()

walk        298887
bus         184140
car         140640
bike        110722
subway       42814
train        23350
airplane         0
Name: modes, dtype: int64

Check on the data leakage using segment level split

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Prepare groups by seg_id_time_split to see the result for protential data leakage
groups_test = df_drop['seg_id_time_split'].astype(str)

# Define X and y
Xt = df_drop  # Example features
yt = df_drop['modes']  # Target variable

# Group-based train/val split
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
for train_idx, val_idx in gss.split(Xt, yt, groups=groups_test):
    Xt_train, Xt_val = Xt.iloc[train_idx], Xt.iloc[val_idx]
    yt_train, yt_val = yt.iloc[train_idx], yt.iloc[val_idx]

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
yt_train_enc = le.fit_transform(yt_train)
yt_val_enc  = le.transform(yt_val)

dict(zip(le.classes_, le.transform(le.classes_)))

{'bike': 0, 'bus': 1, 'car': 2, 'subway': 3, 'train': 4, 'walk': 5}

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = le.classes_

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=yt_train
)

# Map weights to encoded labels
weight_map = dict(zip(le.transform(classes), weights))

sample_weight = [weight_map[k] for k in yt_train_enc]

In [ ]:
from xgboost import XGBClassifier

model  = XGBClassifier(
    n_estimators=350,
    learning_rate=0.1,
    max_depth=25,
    subsample=0.7,
    colsample_bytree=0.6,
    reg_lambda=10,
    objective='multi:softmax',
    num_class=len(le.classes_),
    tree_method='hist',
    eval_metric='mlogloss'
)

model.fit(
    Xt_train[gis_features],
    yt_train_enc,
    sample_weight=sample_weight
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.6, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=25,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=350,
              n_jobs=None, num_class=6, num_parallel_tree=None, ...)

In [ ]:
yt_pred_enc = model.predict(Xt_val[gis_features])
yt_pred = le.inverse_transform(yt_pred_enc)
results.loc['Segment Model',:] = compute_metrics(yt_val,yt_pred)
print(classification_report(yt_val, yt_pred))
print("Macro F1:", f1_score(yt_val, yt_pred, average="macro"))
print("Weighted F1:", f1_score(yt_val, yt_pred, average="weighted"))

              precision    recall  f1-score   support

        bike       0.90      0.79      0.84    127069
         bus       0.79      0.72      0.75    199634
         car       0.81      0.73      0.77    143816
      subway       0.89      0.77      0.83     46751
       train       0.88      0.87      0.88     27607
        walk       0.76      0.91      0.83    278761

    accuracy                           0.80    823638
   macro avg       0.84      0.80      0.82    823638
weighted avg       0.81      0.80      0.80    823638

Macro F1: 0.81561886016455
Weighted F1: 0.8034387240239679


In [ ]:
results.sort_values(by='F1-score (macro avg)', ascending=False)

,Accuracy,F1-score (macro avg),F1-score(weighted avg)
Segment Model,0.804736,0.815619,0.803439
